In [78]:
import pandas as pd

In [79]:
df = pd.read_csv("data/dog_breeds_clean.csv")

In [80]:
df["Exercise Requirements (hrs/day)"].describe()

count    150.000000
mean       1.856667
std        0.453682
min        1.000000
25%        1.500000
50%        2.000000
75%        2.000000
max        3.000000
Name: Exercise Requirements (hrs/day), dtype: float64

In [81]:
df["Grooming Needs"].value_counts()

Grooming Needs
Moderate     51
High         48
Low          37
Very High    14
Name: count, dtype: int64

In [82]:
df["Shedding Level"].value_counts()

Shedding Level
Moderate     87
Low          34
High         28
Very High     1
Name: count, dtype: int64

In [83]:
df["Health Issues Risk"].value_counts()

Health Issues Risk
Moderate    68
Low         59
High        23
Name: count, dtype: int64

In [84]:
# Encoding ordinal variables
grooming_map = {
    "Low": 1,
    "Moderate": 2,
    "High": 3,
    "Very High": 4, 
    "Doesn\'t matter": 5}

shedding_map = {
    "Low": 1,
    "Moderate": 2,
    "High": 3,
    "Very High": 4, 
    "Doesn\'t matter": 5}

health_map = {
    "Low": 1,
    "Moderate": 2,
    "High": 3,  
    "Doesn\'t matter": 4}

df["grooming_score"] = df["Grooming Needs"].map(grooming_map)
df["shedding_score"] = df["Shedding Level"].map(shedding_map)
df["health_risk_score"] = df["Health Issues Risk"].map(health_map)

In [85]:
df.to_csv("data\\dog_breeds_processed.csv", index=False)

In [65]:
def recommend_dogs(size, children, exercise_hours, grooming, shedding, training, health, user_temperament):

    dogs = df.copy()

    # Hard filters
    if size != "Any":
        dogs = dogs[dogs["Size"] == size]

    if children == "Yes":
        dogs = dogs[dogs["Good with Children"] == "Yes"]

    # Exercise match
    dogs["exercise_match"] = dogs["Exercise Requirements (hrs/day)"].apply(lambda x: 1 if x <= exercise_hours else max(0, 1 - (x - exercise_hours)))
    # Grooming needs
    dogs["grooming_match"] = dogs["grooming_score"].apply(lambda x: 1 if x <= grooming else max(0, 1 - (x - grooming) / 3))
    # Shedding tolerance
    dogs["shedding_match"] = dogs["shedding_score"].apply(lambda x: 1 if x <= shedding else max(0, 1 - (x - hedding) / 3))
    # Training
    dogs["training_match"] = dogs["Training Difficulty (1-10)"].apply(lambda x: 1 if x <= training else max(0, 1 - (x - training) / 9))
    # Health issues
    dogs["health_match"] = dogs["health_risk_score"].apply(lambda x: 1 if x <= health else max(0, 1 - (x - health) / 2))

    # Temperament mapping

    temperament_map = {"Affectionate & Social": "affection_sociability",
        "Energetic": "energy_activity",
        "Playful": "playfulness",
        "Calm": "calm_stability",
        "Protective": "protectiveness",
        "Independent": "independence"}

    temperament_columns = [temperament_map[choice] for choice in user_temperament]

    # Temperament match
    if temperament_columns:
        dogs["temperament_match"] = (dogs[temperament_columns].sum(axis=1)/len(temperament_columns))
    else:
        dogs["temperament_match"] = 1

    dogs["overall_match"] = dogs[["exercise_match","grooming_match","shedding_match","training_match","health_match","temperament_match"]].mean(axis=1)
    dogs["overall_match_pct"] = (dogs["overall_match"] * 100).round(1)
    dogs = dogs.sort_values("overall_match",ascending=False)
        
    return dogs

In [74]:
result = recommend_dogs("Medium", "Yes", 3, 2, 4, 10, 1, ["Energetic","Playful","Calm"])

result[["Name", "Size", "exercise_match", "grooming_match", "shedding_match", "training_match", 'health_match', 'temperament_match', 'overall_match_pct']].head(5)

,Name,Size,exercise_match,grooming_match,shedding_match,training_match,health_match,temperament_match,overall_match_pct
69,Ibizan Hound,Medium,1,1.0,1,1,1.0,1.000000,100.0
48,English Foxhound,Medium,1,1.0,1,1,1.0,0.666667,94.4
7,American Foxhound,Medium,1,1.0,1,1,1.0,0.666667,94.4
95,Norwegian Buhund,Medium,1,1.0,1,1,1.0,0.666667,94.4
70,Icelandic Sheepdog,Medium,1,1.0,1,1,1.0,0.666667,94.4


In [71]:
result['Name'].iloc[0]

'Cavalier King Charles Spaniel'

In [72]:
df.columns

Index(['Name', 'Origin', 'Type', 'Unique Feature', 'Friendly Rating (1-10)',
       'Size', 'Grooming Needs', 'Exercise Requirements (hrs/day)',
       'Good with Children', 'Intelligence Rating (1-10)', 'Shedding Level',
       'Health Issues Risk', 'Training Difficulty (1-10)', 'temperament',
       'origin', 'country_code', 'description', 'breed_group', 'history',
       'image', 'weight_min_kg', 'weight_max_kg', 'weight_avg_kg',
       'height_min_cm', 'height_max_cm', 'height_mid_cm',
       'life_span_min_years', 'life_span_max_years', 'life_span_mid_years',
       'temperament list', 'temperament_clean', 'affection_sociability',
       'energy_activity', 'trainability', 'playfulness', 'protectiveness',
       'independence', 'calm_stability', 'confidence_resilience',
       'grooming_score', 'shedding_score', 'health_risk_score'],
      dtype='str')

In [77]:
df[
    (df["Size"] == "Medium") &
    (df["Good with Children"] == "No")
][["Name", "Size", "Good with Children"]]

,Name,Size,Good with Children
38,Chow Chow,Medium,No
119,Saluki,Medium,No
